# 🌿 Leaf Disease Prediction – Training Notebook

**Model:** MobileNetV2 (transfer learning) fine-tuned on PlantVillage dataset  
**Classes:** 38 (plant + disease combinations)  
**Framework:** TensorFlow / Keras

## Steps
1. Mount Google Drive (Colab only)
2. Load dataset from directory
3. Build MobileNetV2 model
4. Train with EarlyStopping + ModelCheckpoint
5. Plot results
6. Save final model

In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"GPU found: {physical_devices}")
else:
    print("No GPU found – running on CPU.")
print("TF version:", tf.__version__)

In [ ]:
# ── 2. Mount Drive (skip if running locally) ──────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = "/content/drive/MyDrive/plant disease"
except ImportError:
    DATASET_PATH = "./dataset"   # local path
    print("Not running on Colab – using local dataset path:", DATASET_PATH)

In [ ]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# ── 4. Dataset paths ──────────────────────────────────────────────────────────
TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VALID_DIR = os.path.join(DATASET_PATH, "valid")

print("Train:", TRAIN_DIR)
print("Valid:", VALID_DIR)

In [ ]:
# ── 5. Data pipeline ──────────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
AUTOTUNE   = tf.data.AUTOTUNE

def normalize(x, y):
    return x / 255.0, y

train_ds = (
    tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        labels='inferred',
        label_mode='categorical',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.keras.utils.image_dataset_from_directory(
        VALID_DIR,
        labels='inferred',
        label_mode='categorical',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

# Store class names before the dataset is consumed
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"{NUM_CLASSES} classes:", class_names[:5], "...")

In [ ]:
# ── 6. Build model ────────────────────────────────────────────────────────────
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False   # freeze base for first phase

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# ── 7. Callbacks ──────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1,
)

elapsed = (time.time() - start) / 60
print(f"\nTraining done in {elapsed:.1f} min")
print(f"Best val accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# ── 9. Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── 10. Save final model ──────────────────────────────────────────────────────
model.save('plant_disease_model_final.h5')
print("Model saved as plant_disease_model_final.h5")

# Optional: copy to Drive
try:
    import shutil
    shutil.copy('plant_disease_model_final.h5',
                '/content/drive/MyDrive/plant_disease_model_final.h5')
    print("Backup saved to Drive.")
except Exception as e:
    print("Drive backup skipped:", e)